# RAG 파이프라인 (Colab, vLLM 서빙)

실행 전 확인: `런타임 > 런타임 유형 변경 > T4 GPU`로 설정.

구성: A 의존성 설치 → B 데이터 업로드 → C schemas → D indexing → E retriever → F augmentation → G generation(vLLM HTTP 위임) → H FastAPI app → I vllm 설치 → J vLLM 엔진 서버 실행(8001) → K RAG 앱 서버 실행(8000) → L 테스트 요청 → M 측정 실행 → N 결과 확인

## A. 의존성 설치

In [ ]:
!pip install -q fastapi "uvicorn[standard]" pydantic httpx \
    langchain-core langchain-community fastembed \
    nest-asyncio

## B. retrospective.md 업로드

In [ ]:
from google.colab import files

uploaded = files.upload()
FILE_PATH = list(uploaded.keys())[0]
print(f"업로드된 파일: {FILE_PATH}")

## C. schemas

In [ ]:
import time
import uuid
from typing import Literal

from pydantic import BaseModel, Field


class ChatMessage(BaseModel):
    role: Literal["user", "assistant", "system"]
    content: str | list[dict]


class ChatCompletionRequest(BaseModel):
    model: str
    messages: list[ChatMessage]
    stream: bool = False
    max_tokens: int | None = None
    max_completion_tokens: int | None = None


class UsageInfo(BaseModel):
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int


class ChatCompletionResponseChoice(BaseModel):
    index: int = 0
    message: ChatMessage
    finish_reason: str = "stop"


class ChatCompletionResponse(BaseModel):
    id: str = Field(default_factory=lambda: f"chatcmpl-{uuid.uuid4().hex}")
    object: str = "chat.completion"
    created: int = Field(default_factory=lambda: int(time.time()))
    model: str
    choices: list[ChatCompletionResponseChoice]
    usage: UsageInfo


class ChatCompletionChunkDelta(BaseModel):
    role: Literal["assistant"] | None = None
    content: str | None = None


class ChatCompletionChunkChoice(BaseModel):
    index: int = 0
    delta: ChatCompletionChunkDelta
    finish_reason: str | None = None


class ChatCompletionChunk(BaseModel):
    id: str
    object: str = "chat.completion.chunk"
    created: int = Field(default_factory=lambda: int(time.time()))
    model: str
    choices: list[ChatCompletionChunkChoice]

## D. indexing (로딩 → 파싱 → 청킹 → 임베딩 → 벡터DB 저장)

In [ ]:
import re
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SECTION_HEADER_PATTERN = re.compile(r"(?=^### )", re.MULTILINE)


def build_vector_store() -> InMemoryVectorStore:
    # 1. 로딩
    markdown_text = Path(FILE_PATH).read_text(encoding="utf-8")

    # 2. 파싱
    raw_sections = SECTION_HEADER_PATTERN.split(markdown_text)

    # 3. 청킹
    section_texts = [section.strip() for section in raw_sections if section.strip().startswith("### ")]

    documents: list[Document] = []
    for section_text in section_texts:
        title_line = section_text.splitlines()[0]
        title = title_line.removeprefix("### ").strip()
        documents.append(Document(page_content=section_text, metadata={"title": title}))

    # 4. 임베딩
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

    # 5. 벡터DB 저장
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(documents)

    return vector_store

## E. retriever

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

TOP_K = 4


def retrieve_relevant_documents(vector_store: InMemoryVectorStore, query: str) -> list[Document]:
    return vector_store.similarity_search(query, k=TOP_K)

## F. augmentation (컨텍스트 구성 → 메시지 구성)

In [ ]:
SYSTEM_PROMPT_TEMPLATE = "아래 참고자료를 바탕으로 질문에 답하시오.\n\n{context}"


def build_augmented_messages(documents: list[Document], user_query: str) -> list[ChatMessage]:
    # 1. 컨텍스트 구성: 검색된 문서를 "[참고자료 N] 제목 + 본문" 형식으로 나열
    context_blocks = []
    for index, document in enumerate(documents, start=1):
        title = document.metadata["title"]
        context_blocks.append(f"[참고자료 {index}] {title}\n{document.page_content}")
    context_text = "\n\n".join(context_blocks)

    # 2. 메시지 구성: 컨텍스트를 담은 system 메시지 + 질문을 담은 user 메시지
    system_message = ChatMessage(role="system", content=SYSTEM_PROMPT_TEMPLATE.format(context=context_text))
    user_message = ChatMessage(role="user", content=user_query)

    return [system_message, user_message]

## G. generation (vLLM 엔진 서버 HTTP 위임)

1단계와 달리 모델을 이 프로세스에 직접 로드하지 않음. 8001번 포트의 vLLM 엔진 서버에 HTTP로 요청을 보내 생성만 위임함.

In [ ]:
import json
from collections.abc import Iterator

import httpx

VLLM_BASE_URL = "http://127.0.0.1:8001"
VLLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 512


def _build_request_payload(messages: list[ChatMessage], max_new_tokens: int, stream: bool) -> dict:
    return {
        "model": VLLM_MODEL_NAME,
        "messages": [message.model_dump() for message in messages],
        "max_tokens": max_new_tokens,
        "stream": stream,
    }


def generate_response(
    messages: list[ChatMessage],
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> tuple[str, int, int]:
    payload = _build_request_payload(messages, max_new_tokens, stream=False)
    response = httpx.post(f"{VLLM_BASE_URL}/v1/chat/completions", json=payload, timeout=120.0)
    response.raise_for_status()
    response_body = response.json()

    response_text = response_body["choices"][0]["message"]["content"]
    prompt_tokens = response_body["usage"]["prompt_tokens"]
    completion_tokens = response_body["usage"]["completion_tokens"]

    return response_text, prompt_tokens, completion_tokens


def stream_response(
    messages: list[ChatMessage],
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> Iterator[str]:
    payload = _build_request_payload(messages, max_new_tokens, stream=True)

    with httpx.stream("POST", f"{VLLM_BASE_URL}/v1/chat/completions", json=payload, timeout=120.0) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line.startswith("data: "):
                continue

            data = line.removeprefix("data: ")
            if data == "[DONE]":
                break

            chunk = json.loads(data)
            delta_content = chunk["choices"][0]["delta"].get("content")
            if delta_content:
                yield delta_content

## H. FastAPI app (lifespan + /v1/chat/completions)

In [ ]:
import uuid
from collections.abc import Iterator
from contextlib import asynccontextmanager

from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.vector_store = build_vector_store()
    yield


app = FastAPI(lifespan=lifespan)


def _extract_text_content(content: str | list[dict]) -> str:
    if isinstance(content, str):
        return content
    return " ".join(part["text"] for part in content if part.get("type") == "text")


def _format_sse_chunk(chunk: ChatCompletionChunk) -> str:
    return f"data: {chunk.model_dump_json()}\n\n"


def _stream_chat_completion_chunks(
    augmented_messages: list[ChatMessage],
    model_name: str,
    max_new_tokens: int,
) -> Iterator[str]:
    chunk_id = f"chatcmpl-{uuid.uuid4().hex}"

    for token_text in stream_response(augmented_messages, max_new_tokens):
        chunk = ChatCompletionChunk(
            id=chunk_id,
            model=model_name,
            choices=[ChatCompletionChunkChoice(delta=ChatCompletionChunkDelta(content=token_text))],
        )
        yield _format_sse_chunk(chunk)

    final_chunk = ChatCompletionChunk(
        id=chunk_id,
        model=model_name,
        choices=[ChatCompletionChunkChoice(delta=ChatCompletionChunkDelta(), finish_reason="stop")],
    )
    yield _format_sse_chunk(final_chunk)
    yield "data: [DONE]\n\n"


@app.post("/v1/chat/completions", response_model=None)
def chat_completions(
    chat_request: ChatCompletionRequest, http_request: Request
) -> ChatCompletionResponse | StreamingResponse:
    user_query = _extract_text_content(chat_request.messages[-1].content)

    documents = retrieve_relevant_documents(http_request.app.state.vector_store, user_query)
    augmented_messages = build_augmented_messages(documents, user_query)

    requested_max_tokens = chat_request.max_completion_tokens
    if requested_max_tokens is None:
        requested_max_tokens = chat_request.max_tokens
    max_new_tokens = requested_max_tokens if requested_max_tokens is not None else MAX_NEW_TOKENS

    if chat_request.stream:
        return StreamingResponse(
            _stream_chat_completion_chunks(augmented_messages, chat_request.model, max_new_tokens),
            media_type="text/event-stream",
        )

    response_text, prompt_tokens, completion_tokens = generate_response(augmented_messages, max_new_tokens)

    response_message = ChatMessage(role="assistant", content=response_text)
    choice = ChatCompletionResponseChoice(message=response_message)
    usage = UsageInfo(
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        total_tokens=prompt_tokens + completion_tokens,
    )
    return ChatCompletionResponse(model=chat_request.model, choices=[choice], usage=usage)

## I. vllm 설치

vLLM 엔진 서버 실행과 측정 도구(vllm bench serve) 둘 다 이 패키지가 필요함. 설치 용량이 크고 시간이 걸림(수 분 이상).

In [ ]:
!pip install -q vllm
!pip uninstall -y -q torchaudio

## J. vLLM 엔진 서버 실행 (8001번 포트)

RAG 앱(8000번)과 별도 프로세스로 띄움. 헬스체크로 최대 240초 대기함.

In [ ]:
import subprocess
import time

import requests

vllm_process = subprocess.Popen(
    [
        "vllm", "serve", "Qwen/Qwen2.5-3B-Instruct",
        "--port", "8001",
        "--dtype", "half",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for _ in range(120):
    time.sleep(2)
    if vllm_process.poll() is not None:
        print("프로세스 종료됨, 종료 코드:", vllm_process.poll())
        print(vllm_process.stdout.read())
        break
    try:
        requests.get("http://127.0.0.1:8001/health", timeout=1)
        print("vLLM 엔진 서버 준비 완료")
        break
    except requests.exceptions.ConnectionError:
        continue
else:
    print("240초 안에 준비되지 않음")

### J-보조1. 더 기다리기 (문제 발생 시)

J 셀이 240초 안에 준비되지 않았지만 프로세스가 죽지는 않았을 때, vllm_process를 새로 띄우지 않고 더 기다림.

In [ ]:
for _ in range(180):
    time.sleep(5)
    if vllm_process.poll() is not None:
        print("프로세스 종료됨, 종료 코드:", vllm_process.poll())
        print(vllm_process.stdout.read())
        break
    try:
        requests.get("http://127.0.0.1:8001/health", timeout=1)
        print("vLLM 엔진 서버 준비 완료")
        break
    except requests.exceptions.ConnectionError:
        continue
else:
    print("여전히 준비되지 않음")

### J-보조2. vLLM 엔진 서버 로그 확인 (문제 발생 시)

J, J-보조1 실행 후에도 준비되지 않았을 때, 프로세스가 지금까지 남긴 로그를 논블로킹으로 확인함.

In [ ]:
import fcntl
import os

fd = vllm_process.stdout.fileno()
flags = fcntl.fcntl(fd, fcntl.F_GETFL)
fcntl.fcntl(fd, fcntl.F_SETFL, flags | os.O_NONBLOCK)

try:
    print(vllm_process.stdout.read())
except Exception:
    print("아직 출력 없음")

## K. RAG 앱 서버 실행 (8000번 포트)

In [ ]:
import time
from threading import Thread

import nest_asyncio
import requests
import uvicorn

nest_asyncio.apply()


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)


server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

for _ in range(120):
    time.sleep(2)
    try:
        requests.get("http://127.0.0.1:8000/docs", timeout=1)
        print("RAG 앱 서버 준비 완료")
        break
    except requests.exceptions.ConnectionError:
        continue
else:
    print("RAG 앱 서버가 240초 안에 준비되지 않음. 로그 확인 필요")

## L. 테스트 요청

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-3B-Instruct",
        "messages": [{"role": "user", "content": "docker run 관련 트러블슈팅 있었나?"}],
    },
)
print(response.status_code)
print(response.json())

## M. 측정 실행

부하 조건은 1단계와 동일하게 고정함(Dataset random, Num Prompts 20, Max Concurrency 4, Request Rate inf, Input/Output Length 128).

In [ ]:
!mkdir -p ./results

!vllm bench serve \
    --backend openai-chat \
    --base-url http://127.0.0.1:8000 \
    --endpoint /v1/chat/completions \
    --model "Qwen/Qwen2.5-3B-Instruct" \
    --dataset-name random \
    --num-prompts 20 \
    --max-concurrency 4 \
    --request-rate inf \
    --random-input-len 128 \
    --random-output-len 128 \
    --ignore-eos \
    --percentile-metrics ttft,tpot,itl,e2el \
    --save-result \
    --result-dir ./results \
    --result-filename vllm_result.json

## N. 측정 결과 확인

In [ ]:
import json

with open("./results/vllm_result.json", "r", encoding="utf-8") as f:
    vllm_result = json.load(f)

print("Request throughput (req/s):", vllm_result.get("request_throughput"))
print("Output token throughput (tok/s):", vllm_result.get("output_throughput"))
print("Mean TTFT (ms):", vllm_result.get("mean_ttft_ms"))
print("Mean E2E Latency (ms):", vllm_result.get("mean_e2el_ms"))